# Fine-tune LightOnOCR for Sinhala OCR

This notebook fine-tunes LightOnOCR-1B on a custom Sinhala OCR dataset.

Dataset Statistics:
- Train: 707 samples
- Validation: 101 samples
- Test: 202 samples

Total: 1010 samples

## 1. Installation

Install required libraries including the transformers fork with LightOnOCR support.

In [ ]:
!pip install -q -U datasets accelerate peft
!pip install -q transformers==5.0.0
!pip install -q huggingface-hub==1.3.1
!pip install -U bitsandbytes>=0.46.1
!pip install -q jiwer
!pip install Pillow==11.3.0

## 2. Load Dataset from Hugging Face

Load your Sinhala OCR dataset uploaded to Hugging Face.

In [ ]:
from datasets import load_dataset
from huggingface_hub import login
import pandas as pd
from PIL import Image

# Login to Hugging Face (if private dataset)
login()

In [ ]:
# Load your uploaded dataset
HF_USERNAME = "avishadilhara"
DATASET_NAME = "sinhala-ocr-lk-acts-1010"

print(f"Loading dataset from Hugging Face...")
dataset = load_dataset(f"{HF_USERNAME}/{DATASET_NAME}")

print(f"Dataset loaded successfully!")
print(f"   Train: {len(dataset['train'])} samples")
print(f"   Validation: {len(dataset['eval'])} samples")
print(f"   Test: {len(dataset['test'])} samples")

# Preview sample
sample = dataset['train'][0]
print(f"\nSample structure:")
print(f"   Image size: {sample['image'].size}")
print(f"   Text length: {len(sample['text'])} chars")
print(f"   Year: {sample.get('year', 'N/A')}")
print(f"   Preview: {sample['text'][:100]}...")

## 3. Load Model and Processor

Load the LightOnOCR model with the processor for image and text handling.

In [ ]:
import torch
from transformers import LightOnOcrProcessor


# Model checkpoint
model_id = "lightonai/LightOnOCR-2-1B"

device = "cuda:0" if torch.cuda.is_available() else "cpu"

processor = LightOnOcrProcessor.from_pretrained(model_id)
processor.tokenizer.padding_side = "left"

print(f"Using device: {device}")

## 4. Load Model (Full Fine-tuning)

Load the model with optional layer freezing to reduce memory requirements.

In [ ]:
from transformers import LightOnOcrForConditionalGeneration

model = LightOnOcrForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    attn_implementation="sdpa",
    device_map="cuda:0" if torch.cuda.is_available() else "cpu",
).to(device)

# Train vision encoder and projection (adapt to Sinhala script)
for param in model.model.vision_encoder.parameters():
    param.requires_grad = True
print(f"Vision encoder trainable: True")

for param in model.model.vision_projection.parameters():
    param.requires_grad = True
print(f"Vision projection trainable: True")

# FREEZE language model — matches official notebook, prevents gradient instability
# The LM already knows how to generate text; we only need vision to learn Sinhala script
# for param in model.model.language_model.parameters():
#     param.requires_grad = False
# print(f"Language model frozen: True")

# Count trainable params
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"\nTrainable parameters: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")

In [ ]:
from transformers import LightOnOcrForConditionalGeneration, BitsAndBytesConfig
from peft import get_peft_model, LoraConfig, prepare_model_for_kbit_training
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

model = LightOnOcrForConditionalGeneration.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    quantization_config=bnb_config
)

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",],
    lora_dropout=0.1,  # Increased dropout for larger rank to prevent overfitting
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print("Model loaded with 4-bit quantization + LoRA")

## 5. Prepare Dataset Collator

Create a data collator to format batches for training.

In [ ]:
ASSISTANT_START_PATTERN = [151645, 198, 151644, 77091, 198]
MAX_LENGTH = 3072  # Increased to 3072 — Sinhala legal texts need more token space to avoid truncation
LONGEST_EDGE = 1024

def collate_fn(examples):
    batch_messages = []
    batch_images = []

    for example in examples:
        image = example['image'].convert('RGB')
        batch_images.append(image)

        # Truncate very long text to avoid all labels being masked out
        # Leave room for image tokens + prompt tokens (~500 tokens)
        assistant_text = example['text'].strip()
        # Cap text length to prevent token overflow — adjust if needed
        if len(assistant_text) > 3000:
            assistant_text = assistant_text[:3000]

        messages = [
            {"role": "user", "content": [{"type": "image"}]},
            {"role": "assistant", "content": [{"type": "text", "text": assistant_text}]},
        ]
        batch_messages.append(messages)

    if len(batch_images) == 0:
        return None

    texts = [
        processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
        for messages in batch_messages
    ]

    inputs = processor(
        text=texts,
        images=batch_images,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        size={"longest_edge": LONGEST_EDGE},
    )

    labels = inputs["input_ids"].clone()
    pad_token_id = processor.tokenizer.pad_token_id

    valid_label_counts = []  # Track how many valid labels per sample

    for i in range(len(labels)):
        full_ids = inputs["input_ids"][i].tolist()

        assistant_content_start = None
        for idx in range(len(full_ids) - len(ASSISTANT_START_PATTERN)):
            if (
                full_ids[idx: idx + len(ASSISTANT_START_PATTERN)]
                == ASSISTANT_START_PATTERN
            ):
                assistant_content_start = idx + len(ASSISTANT_START_PATTERN)
                break

        if assistant_content_start is None:
            print(f"Warning: Could not find assistant marker in sample {i}")
            labels[i, :] = -100
            valid_label_counts.append(0)
        else:
            labels[i, :] = -100
            count = 0
            for idx in range(assistant_content_start, len(full_ids)):
                if full_ids[idx] == pad_token_id:
                    break
                labels[i, idx] = inputs["input_ids"][i, idx]
                count += 1
            valid_label_counts.append(count)

    labels[inputs["input_ids"] == pad_token_id] = -100
    inputs["labels"] = labels
    inputs["pixel_values"] = inputs["pixel_values"].to(torch.bfloat16)

    # Safety check: if NO valid labels in entire batch, skip batch to avoid NaN
    total_valid = sum(valid_label_counts)
    if total_valid == 0:
        print(f"WARNING: Batch has 0 valid labels! All text was truncated. Returning None.")
        return None

    # Log warning for samples with very few valid labels
    for i, count in enumerate(valid_label_counts):
        if count == 0:
            print(f"Warning: Sample {i} has 0 valid label tokens (text fully truncated)")
        elif count < 10:
            print(f"Warning: Sample {i} has only {count} valid label tokens")

    return inputs

## 6. Test the Collator

Verify the collator works correctly with a small batch.

In [ ]:
# Test with a small batch
test_batch = collate_fn([dataset['train'][0], dataset['train'][1]])
print("Input shape:", test_batch["input_ids"].shape)
print("Labels shape:", test_batch["labels"].shape)
print("Pixel values shape:", test_batch["pixel_values"].shape)

## 7. Training Configuration

Set up training arguments and trainer.

In [ ]:
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
torch.set_float32_matmul_precision('high')

output_dir = "./lightonocr-sinhala-finetuned"

training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=30,              # Early stopping (patience=5) will stop earlier
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,    # Effective batch size = 16 for smoother gradients
    learning_rate=5e-5,                 # Lower LR for stable QLoRA convergence
    lr_scheduler_type="cosine",
    warmup_steps=50,                # Adaptive warmup based on dataset size

    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=5,

    weight_decay=0.05,                # Stronger regularization

    fp16=False,
    bf16=True,

    gradient_checkpointing=True,
    max_grad_norm=1.0,
    optim="adamw_8bit",               # Memory-efficient optimizer

    dataloader_pin_memory=False,
    dataloader_num_workers=0,

    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",

    seed=3407,
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['eval'],
    data_collator=collate_fn,
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=5),
    ]
)

print("Trainer initialized successfully!")
print(f"Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")

## 8. Start Training

Begin the fine-tuning process.

In [ ]:
print("Starting training...")
trainer.train()
print("Training complete!")

In [ ]:
print("Starting training...")
trainer.train(resume_from_checkpoint="/kaggle/working/lightonocr-sinhala-finetuned/checkpoint-630")
print("Training complete!")

In [ ]:
import json

with open("/kaggle/working/lightonocr-sinhala-finetuned/checkpoint-630/trainer_state.json", "r") as f:
    state = json.load(f)

df = pd.DataFrame(state["log_history"])
train_logs = df[df["loss"].notna()][["epoch", "step", "loss", "learning_rate"]].copy()
train_logs.columns = ["Epoch", "Step", "Training Loss", "Learning Rate"]
eval_logs = df[df["eval_loss"].notna()][["epoch", "step", "eval_loss"]].copy()
eval_logs.columns = ["Epoch", "Step", "Validation Loss"]
merged = pd.merge(train_logs, eval_logs, on=["Epoch", "Step"], how="outer").sort_values("Step")
print("Training & Validation Loss History")
display(merged.reset_index(drop=True))

In [ ]:
import shutil
import os
from IPython.display import FileLink

checkpoint_path = ""
zip_name = "checkpoint-135"

if os.path.exists(checkpoint_path):
    if os.path.exists(f"{zip_name}.zip"):
        os.remove(f"{zip_name}.zip")

    shutil.make_archive(zip_name, 'zip', checkpoint_path)

    print(f"Created {zip_name}.zip")
    print(f"Size: {os.path.getsize(f'{zip_name}.zip') / (1024*1024):.2f} MB")

    display(FileLink(f"{zip_name}.zip"))
else:
    print(f"Checkpoint not found: {checkpoint_path}")

## 9. Save the Model

Save the fine-tuned model and processor.

In [ ]:
import torch
from transformers import LightOnOcrProcessor, LightOnOcrForConditionalGeneration, BitsAndBytesConfig
from peft import PeftModel

model_id = "lightonai/LightOnOCR-2-1B"
best_checkpoint_path = "/kaggle/working/lightonocr-sinhala-finetuned/checkpoint-630"
save_path = "./lightonocr-sinhala-finetuned-Qlora"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

print("Loading base model...")
base_model = LightOnOcrForConditionalGeneration.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16,
    quantization_config=bnb_config
)

processor = LightOnOcrProcessor.from_pretrained(model_id)

print(f"Loading best checkpoint from {best_checkpoint_path}...")
model = PeftModel.from_pretrained(base_model, best_checkpoint_path)
model.eval()

model.save_pretrained(save_path)
processor.save_pretrained(save_path)
print(f"Best model (checkpoint-630) saved to {save_path}")

In [ ]:
trainer.save_model("./lightonocr-sinhala-finetuned-Qlora")
processor.save_pretrained("./lightonocr-sinhala-finetuned-Qlora")
print(f"Model saved to {"./lightonocr-sinhala-finetuned-Qlora"}")

In [ ]:
import shutil
import os
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
zip_filename = f"lightonocr_sinhala_model_{timestamp}"

print(f"Zipping model directory: {"./lightonocr-sinhala-finetuned-Qlora"}")
print("This may take a few minutes...")

shutil.make_archive(zip_filename, 'zip', "./lightonocr-sinhala-finetuned-Qlora")

zip_path = f"{zip_filename}.zip"
zip_size_mb = os.path.getsize(zip_path) / (1024 * 1024)

print(f"Zip file created successfully!")
print(f"Filename: {zip_path}")
print(f"Size: {zip_size_mb:.2f} MB")

## 10. Inference Function

Create a function to run inference on new images.

In [ ]:
def run_inference(image):
    messages = [
        {"role": "user", "content": [{"type": "image"}]},
    ]

    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    inputs = processor(
        text=text,
        images=[image],
        return_tensors="pt",
        size={"longest_edge": LONGEST_EDGE},
    ).to(device)

    inputs["pixel_values"] = inputs["pixel_values"].to(torch.float32)

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=2048,
            do_sample=False,
        )

    generated_text = processor.batch_decode(
        generated_ids, skip_special_tokens=True
    )[0]

    return generated_text

## 11. Test on Validation Sample

Test the fine-tuned model on a validation sample.

In [ ]:
# Test on a validation sample
test_idx = 0
test_sample = dataset['eval'][test_idx]
test_image = test_sample['image']

print("Running inference...")
result = run_inference(test_image)

print("\nGenerated Text:")
print(result)

print("\nGround Truth:")
print(test_sample['text'])

display(test_image)

## 12. Evaluate on Test Set

Calculate Character Error Rate (CER) on the test set.

In [ ]:
from jiwer import cer

print("Evaluating on test set...")

predictions = []
references = []

for i in range(min(50, len(dataset['test']))):
    sample = dataset['test'][i]
    image = sample['image']
    ground_truth = sample['text']

    result = run_inference(image)

    predictions.append(result)
    references.append(ground_truth)

    if (i + 1) % 10 == 0:
        print(f"Processed {i + 1} samples...")

error_rate = cer(references, predictions) * 100
print(f"\nCharacter Error Rate (CER): {error_rate:.2f}%")

## 13. Upload to Hugging Face Hub (Optional)

Push the fine-tuned model to Hugging Face Hub for sharing.

In [ ]:
from huggingface_hub import notebook_login, logout

# Logout and re-login for push permissions
logout()
notebook_login()

# Push to Hub
hub_model_id = "avishadilhara/lightonocr-sinhala-finetuned"
trainer.push_to_hub(hub_model_id)
processor.push_to_hub(hub_model_id)

print(f"Model pushed to https://huggingface.co/{hub_model_id}")